# Musanga — Upload UAV RGB and probability maps to Google Earth Engine

**Cloud project:** `musanga-511110` · **Account:** run Colab with `kristof.vanoost@uclouvain.be` (Colab Pro and source Drive); authenticate Earth Engine with that account (granted access to Musanga).

This Colab uploads the original georeferenced UAV RGB orthomosaics and the **KVO_04 multisite U-Net probability maps** to Earth Engine. The separate [GEE viewer script](https://github.com/kvanoost/Musanga/blob/main/gee/Musanga_UAV_probability_viewer.js) adds interactive RGB, probability and **binary threshold overlays**. Binary layers are computed inside Earth Engine from the probability asset, so there is no need to upload a second giant raster.

**Source encoding:** probability GeoTIFF is `uint16`; 0–10000 means 0–1 probability, `65535` is NoData. **Caution:** the KVO_04 workflow may mask probability outputs to designated valid ROIs, so these are not necessarily wall-to-wall predictions over the entire RGB orthomosaic.

The files can be large. Upload uses a **temporary Google Cloud Storage bucket** in the Musanga Cloud project, then Earth Engine image ingestion. Cloud Storage charges and EE quota/permission requirements may apply. Originals on Drive are never changed. Re-running skips existing GCS objects of matching byte size and existing EE assets; ingestion itself is asynchronous. Google authentication and billing/project permissions are required. No files are uploaded until you execute the upload cell.


In [ ]:
# Keep Colab's pinned authentication library; avoid '-U' upgrading google-auth.
%pip -q install "google-auth==2.49.0" earthengine-api google-cloud-storage rasterio
# If you already installed incompatible versions in this session, restart the runtime once.


In [ ]:
from pathlib import Path
import google.auth
from google.colab import auth, drive
import ee, rasterio
from google.cloud import storage

CLOUD_PROJECT = "musanga-511110"
ACCOUNT = "kristof.vanoost@uclouvain.be"  # Use Colab Pro / UCLouvain credentials
BUCKET_NAME = "musanga-511110-musanga-uav-stage"  # Must be globally unique; change if taken
ASSET_FOLDER = f"projects/{CLOUD_PROJECT}/assets/musanga_uav"

auth.authenticate_user()
creds, _ = google.auth.default(scopes=[
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/earthengine"
])
ee.Initialize(credentials=creds, project=CLOUD_PROJECT)
client = storage.Client(project=CLOUD_PROJECT, credentials=creds)
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
GEO = ROOT/"data"/"Geo_Ref"
MAPS = ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"
print("Earth Engine project:", CLOUD_PROJECT)
print("Expected Google account:", ACCOUNT)
print("Asset folder:", ASSET_FOLDER)

In [ ]:
# Enable Baego first; add sites when ready.
SELECTED_SITES = ["Baego"]
# SELECTED_SITES = ["Baego", "Yoko", "Yangambi", "MOS_1_2"]

SITE_RASTERS = {
    "Baego": "Baego_georef.tif",
    "Yoko": "Yoko_kvo_georef.tif",
    "Yangambi": "Yangambi_georef.tif",
    "MOS_1_2": "MOS_1_2_kvo.tif",
}

def inspect_site(site):
    rgb=GEO/SITE_RASTERS[site]
    prob=MAPS/f"{site}_musanga_probability_uint16.tif"
    for label,path in [("RGB",rgb),("probability",prob)]:
        if not path.exists():
            raise FileNotFoundError(f"{site} {label} missing: {path}")
        with rasterio.open(path) as src:
            print(site,label, "bands",src.count,"dtype",src.dtypes,"CRS",src.crs,
                  "resolution",src.res,"NoData",src.nodata,
                  "size GB",round(path.stat().st_size/1e9,2))
            assert src.crs is not None, f"Missing CRS: {path}"
            if label=="RGB":
                assert src.count>=3, f"RGB needs at least 3 bands: {path}"
            else:
                assert src.count==1 and src.dtypes[0]=="uint16", f"Expected one uint16 probability band: {path}"
    return rgb,prob

SOURCES={site:inspect_site(site) for site in SELECTED_SITES}
print("Inputs verified. Original GeoTIFFs remain unchanged.")

In [ ]:
import json

# Bucket creation may require the Storage Admin role and billing.
# If a bucket is already available, set BUCKET_NAME above to that bucket instead.
bucket=client.lookup_bucket(BUCKET_NAME)
if bucket is None:
    try:
        bucket=client.create_bucket(BUCKET_NAME,location="EU")
        print("Created Cloud Storage bucket:", bucket.name)
    except Exception as exc:
        raise RuntimeError(
            "Could not create the GCS staging bucket. Check project billing/Storage permissions "
            "or supply the name of an existing bucket you can write to."
        ) from exc
else:
    print("Using existing bucket:",bucket.name)

try:
    ee.data.getAsset(ASSET_FOLDER)
    print("Existing EE folder:",ASSET_FOLDER)
except ee.EEException as err:
    if "not found" not in str(err).lower() and "does not exist" not in str(err).lower():
        raise
    ee.data.createAsset({"type":"FOLDER"},ASSET_FOLDER)
    print("Created Earth Engine folder:",ASSET_FOLDER)

In [ ]:
# Upload to Cloud Storage, then submit two EE ingestion tasks per site.
# START_INGESTION=True submits tasks but does NOT wait for them to complete.
START_INGESTION=True

def asset_exists(asset_id):
    try:
        ee.data.getAsset(asset_id)
        return True
    except ee.EEException as exc:
        if "not found" in str(exc).lower() or "does not exist" in str(exc).lower():
            return False
        raise

def stage_file(local_path, object_name):
    blob=bucket.blob(object_name)
    size=local_path.stat().st_size
    if blob.exists(client=client):
        blob.reload(client=client)
        if blob.size==size:
            print("GCS already staged:",object_name)
            return f"gs://{bucket.name}/{object_name}"
        raise ValueError(f"GCS object exists but has different size: {object_name}; choose a new name.")
    print(f"Staging {local_path.name} ({size/1e9:.2f} GB)...")
    blob.chunk_size=8*1024*1024
    blob.upload_from_filename(str(local_path),timeout=3600,checksum="auto")
    print("Staged:",object_name)
    return f"gs://{bucket.name}/{object_name}"

submitted=[]
for site,(rgb_path,prob_path) in SOURCES.items():
    for kind,path in [("RGB",rgb_path),("Musanga_probability",prob_path)]:
        asset=f"{ASSET_FOLDER}/{site}_{kind}"
        if asset_exists(asset):
            print("Existing EE asset, skipping:",asset)
            continue
        gs_uri=stage_file(path,f"musanga_uav/{site}/{path.name}")
        manifest={"name":asset,
                  "tilesets":[{"id":"t0","sources":[{"uris":[gs_uri]}]}],
                  "properties":{"site":site,"dataset":"Musanga UAV","source_file":path.name}}
        if kind=="RGB":
            manifest["bands"]=[
                {"id":label,"tilesetId":"t0","tilesetBandIndex":i,"pyramidingPolicy":"MEAN"}
                for i,label in enumerate(["red","green","blue"])
            ]
        else:
            manifest["bands"]=[{"id":"probability_uint16","tilesetId":"t0",
                "tilesetBandIndex":0,"missingData":{"values":[65535]},
                "pyramidingPolicy":"MEAN"}]
            manifest["properties"]["probability_scale"]=10000
        if START_INGESTION:
            result=ee.data.startIngestion(None,manifest)
            submitted.append(dict(site=site,kind=kind,asset=asset,result=result))
            print("Submitted EE ingestion:",asset,result)
        else:
            print("DRY RUN manifest:",json.dumps(manifest,indent=2))
print("Tasks submitted:",len(submitted))
print("Note: upload completion is separate from Earth Engine asset ingestion completion.")

In [ ]:
# Run this cell again later to check progress. The uploads are run by Earth Engine.
import pandas as pd
task_rows=[]
for t in ee.data.getTaskList():
    desc=t.get("description","")
    task_id=t.get("id","")
    # Ingestion descriptions may omit the asset ID; show all ingestion tasks.
    if t.get("task_type")=="INGEST" or t.get("type")=="INGEST" or "Musanga" in desc:
        task_rows.append({k:t.get(k) for k in ("id","state","description","error_message")})
display(pd.DataFrame(task_rows))
for site in SELECTED_SITES:
    for suffix in ("RGB","Musanga_probability"):
        asset=f"{ASSET_FOLDER}/{site}_{suffix}"
        print(site,suffix,"READY" if asset_exists(asset) else "not yet available")
print("Once both site assets are READY, open the viewer script in the GEE Code Editor.")

## Viewer

Open the source [Musanga UAV probability viewer](https://github.com/kvanoost/Musanga/blob/main/gee/Musanga_UAV_probability_viewer.js), copy it to the [GEE Code Editor](https://code.earthengine.google.com/) and press **Run**. The control panel includes site selection, binary threshold, transparency, and layer toggles.

To share with colleagues: grant them access to the Earth Engine assets (or share via an Earth Engine App) and make sure they can access the Cloud project. Merely sending the JavaScript link does not grant access to private assets.
